# Black-Box Optimisation (BBO) Capstone Project
**Cohort:** IMP-PCMLAI-26-06  
**Author:** Kalyan Logisetti  

## Overview
This notebook implements an iterative Bayesian Optimisation framework across 8 synthetic black-box functions ranging from 2D to 8D. Each weekly iteration fits Gaussian Process (GP) surrogates using Matérn kernels with Automatic Relevance Determination (ARD) and evaluates an Upper Confidence Bound (UCB) acquisition function to select new query vectors.

## 1. Environment Setup & Data Loading
In this section, we import the required Python libraries, extract the initial dataset archive (`Initial_data_points_starter.zip`), and load the observations for all 8 synthetic functions into memory.

In [10]:
import os
import numpy as np
import pandas as pd

# Define expected dimensions for Functions 1 through 8
FUNCTION_SPECS = {
    "Function 1": 2,
    "Function 2": 2,
    "Function 3": 3,
    "Function 4": 4,
    "Function 5": 4,
    "Function 6": 5,
    "Function 7": 6,
    "Function 8": 8,
}

# Dictionary to hold the loaded arrays
data_dict = {}

# Load starter data for each function subfolder
for i in range(1, 9):
    func_key = f"Function {i}"
    x_path = f"initial_data/function_{i}/initial_inputs.npy"
    y_path = f"initial_data/function_{i}/initial_outputs.npy"

    X = np.load(x_path)
    y = np.load(y_path)

    data_dict[func_key] = {
        "X": X,
        "y": y
    }

# Display summary table of loaded datasets
summary_data = []
for func_name, spec in FUNCTION_SPECS.items():
    X_shape = data_dict[func_name]["X"].shape
    y_shape = data_dict[func_name]["y"].shape
    summary_data.append({
        "Function": func_name,
        "Expected Dim": f"{spec}D",
        "X Shape (Inputs)": str(X_shape),
        "y Shape (Outputs)": str(y_shape),
        "Sample Input (Row 0)": np.array2string(data_dict[func_name]["X"][0], precision=4, suppress_small=True)
    })

summary_df = pd.DataFrame(summary_data)
print("Data Loading Complete. Summary Table:")
summary_df

Data Loading Complete. Summary Table:


,Function,Expected Dim,X Shape (Inputs),y Shape (Outputs),Sample Input (Row 0)
0,Function 1,2D,"(10, 2)","(10,)",[0.3194 0.763 ]
1,Function 2,2D,"(10, 2)","(10,)",[0.6658 0.124 ]
2,Function 3,3D,"(15, 3)","(15,)",[0.1715 0.3439 0.2487]
3,Function 4,4D,"(30, 4)","(30,)",[0.897 0.7256 0.1754 0.7017]
4,Function 5,4D,"(20, 4)","(20,)",[0.1914 0.0382 0.6074 0.4146]
5,Function 6,5D,"(20, 5)","(20,)",[0.7282 0.1547 0.7326 0.694 0.0564]
6,Function 7,6D,"(30, 6)","(30,)",[0.2726 0.3245 0.8971 0.833 0.1541 0.7959]
7,Function 8,8D,"(40, 8)","(40,)",[0.605 0.2922 0.9085 0.3555 0.2017 0.5753 0.3...


## 2. Surrogate Modeling & Acquisition Strategy

To model the unknown black-box functions, we fit a **Gaussian Process (GP) Regressor** with a **Matérn 2.5 ARD kernel**. The Automatic Relevance Determination (ARD) allows the model to learn individual feature weights and length-scales across each dimension.

To guide the selection of the next evaluation points, we maximize the **Upper Confidence Bound (UCB)** acquisition function:

$$UCB(x) = \mu(x) + \beta \cdot \sigma(x)$$

Where:
- $\mu(x)$ represents predicted mean reward (Exploitation).
- $\sigma(x)$ represents predicted model uncertainty/variance (Exploration).
- $\beta = 2.0$ controls the exploration-exploitation balance.

We optimize $-UCB(x)$ using multi-start **L-BFGS-B** within the bounded hypercube domain $[0, 1]^d$.

In [12]:
# Multi-start L-BFGS-B optimization for UCB Acquisition across all 8 functions

def generate_weekly_queries(data_dict, beta=2.0, random_state=42):
    np.random.seed(random_state)
    query_results = {}

    for func_name, dim in FUNCTION_SPECS.items():
        X = np.array(data_dict[func_name]['X'])
        y = np.array(data_dict[func_name]['y']).ravel()

        # 1. Fit Gaussian Process with Matérn 2.5 ARD Kernel
        kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=np.ones(dim), length_scale_bounds=(1e-2, 1e2), nu=2.5)
        gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=3, random_state=random_state)
        gp.fit(X, y)

        # 2. Define Negative UCB Acquisition Function
        def neg_ucb(x_vec):
            x_vec = np.atleast_2d(x_vec)
            mean, std = gp.predict(x_vec, return_std=True)
            return -(mean[0] + beta * std[0])

        # 3. Multi-start L-BFGS-B Optimization
        best_x = None
        best_val = np.inf
        bounds = [(0.0, 1.0)] * dim

        for _ in range(15):
            x0 = np.random.uniform(0.0, 1.0, dim)
            res = minimize(neg_ucb, x0, bounds=bounds, method='L-BFGS-B')
            if res.fun < best_val:
                best_val = res.fun
                best_x = res.x

        # 4. Format query as hyphen-separated 6-decimal float string
        formatted_str = "-".join([f"{val:.6f}" for val in best_x])
        query_results[func_name] = formatted_str
        print(f"{func_name} ({dim}D): {formatted_str}")

    return query_results

print("--- GENERATING WEEK 2 QUERY CANDIDATES ---")
week2_queries = generate_weekly_queries(data_dict, beta=2.0)

--- GENERATING WEEK 2 QUERY CANDIDATES ---


/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 1 (2D): 1.000000-1.000000
Function 2 (2D): 0.956801-1.000000
Function 3 (3D): 1.000000-0.000000-0.755108


/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 3 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 4 (4D): 0.896114-0.330454-0.110024-0.227935


...

Function 5 (4D): 0.241723-0.819247-0.889061-0.881089
Function 6 (5D): 0.179615-0.000000-0.148792-1.000000-0.000000


/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 7 (6D): 0.000000-0.000000-1.000000-0.160841-0.342078-0.694080


/usr/local/lib/python3.13/dist-packages/sklearn/gaussian_process/kernels.py:452: ConvergenceWarning: The optimal value found for dimension 7 of parameter k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Function 8 (8D): 0.000000-0.000000-0.107377-0.000000-1.000000-0.273015-0.000000-0.000000


In [13]:
import warnings
from sklearn.exceptions import ConvergenceWarning

# Suppress hyperparameter boundary convergence warnings for clean presentation
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Format into a clean Pandas DataFrame for submission/portfolio display
results_df = pd.DataFrame([
    {"Function": name, "Dimension": f"{FUNCTION_SPECS[name]}D", "Week 2 Query Point": query}
    for name, query in week2_queries.items()
])

print("=== WEEK 2 SUBMISSION QUERIES ===")
results_df

=== WEEK 2 SUBMISSION QUERIES ===


,Function,Dimension,Week 2 Query Point
0,Function 1,2D,1.000000-1.000000
1,Function 2,2D,0.956801-1.000000
2,Function 3,3D,1.000000-0.000000-0.755108
3,Function 4,4D,0.896114-0.330454-0.110024-0.227935
4,Function 5,4D,0.241723-0.819247-0.889061-0.881089
5,Function 6,5D,0.179615-0.000000-0.148792-1.000000-0.000000
6,Function 7,6D,0.000000-0.000000-1.000000-0.160841-0.342078-0...
7,Function 8,8D,0.000000-0.000000-0.107377-0.000000-1.000000-0...


### Strategy Reflection & Observations (Week 2)

- **Model Fitting:** Gaussian Process models were successfully fit across all 8 dimensions using Matérn 2.5 ARD kernels to handle varying feature sensitivity across parameters.
- **Exploration vs. Exploitation ($\beta = 2.0$):** The UCB acquisition function naturally prioritized regions with high epistemic uncertainty, driving several queries toward the domain boundaries ($0.0$ and $1.0$) where initial data points were sparse.
- **Next Steps:** Once the evaluations for these Week 2 query points are returned, we will append $(x_{new}, y_{new})$ to `data_dict` and update our GP surrogates for the Week 3 iteration.